# 机器翻译作业总结

## 1. 本次代码的目标

本次代码是在 0914 基础 RNN Seq2Seq 的基础上，完成一个更完整的中英机器翻译工程流程：

```text
中文句子 → 双向 GRU Encoder → 各层正反向最终状态 → 单向 GRU Decoder → 英文句子
```

本次代码的主要目标如下：

1. 读取中英文平行语料，完成分词、编码和动态 Padding；
2. 只使用训练集构建中英文词表，避免验证集信息泄漏；
3. 使用双层双向 GRU 构建 Encoder，使用双层单向 GRU 构建 Decoder；
4. 理解正向、反向 hidden 的拼接和 Encoder-Decoder 状态交接；
5. 在训练中逐步降低 Teacher Forcing 比率；
6. 使用 AdamW、梯度裁剪、Early Stopping 和 TensorBoard 完成训练管理；
7. 分清教师强制条件下的 `val_loss` 与自回归生成质量；
8. 使用贪心解码完成独立翻译，并用真实失败现象继续理解 Attention 的必要性。

这一版虽然比第一版的单层普通 RNN 更完整，但仍然是**无 Attention 的 GRU Seq2Seq**，不是 Transformer。

## 2. 数据预处理 data.py

数据预处理流程如下：

```text
读取 cmn.txt 中英句对
→ 中文统一转成简体并用 jieba 分词
→ 英文使用 NLTK TreebankWordTokenizer 分词
→ 固定随机种子打乱并划分 train/val
→ 只使用训练集构建中文、英文词表
→ Token 编码为整数编号
```

四个特殊 Token 的作用：

| Token | 编号 | 作用 |
| --- | ---: | --- |
| `<PAD>` | 0 | 将同一 Batch 中不同长度的序列补齐 |
| `<SOS>` | 1 | 通知 Decoder 开始生成 |
| `<EOS>` | 2 | 表示一句话结束 |
| `<UNK>` | 3 | 表示词表中不存在的 Token |

当前语料共有 26,388 对中英文句子。按 `TRAIN_VAL_RATE=0.9` 和 `SEED=10` 划分后：

```text
训练集：23,749 对
验证集： 2,639 对
```

当前保存的中文词表大小为 11,250，英文词表大小为 5,028。中英文词表相互独立，大小不同是正常现象。

需要注意：代码在 `zh.json`、`en.json` 不存在时才重新构建词表。因此如果修改数据划分、随机种子或语料，应同步重新生成对应词表，不能让旧词表与新数据划分混用。

## 3. 构建数据集与 DataLoader data.py

`TranslationDataset` 在初始化时完成分词和编码，避免每个 epoch 重复运行分词器。

Encoder 输入为：

```text
中文内容 + <EOS>
```

目标序列为：

```text
<SOS> + 英文内容 + <EOS>
```

假设完整目标是：

```text
<SOS> I love you <EOS>
```

那么 Decoder 的逐步训练关系为：

```text
输入：<SOS> I    love you
标签：      I love  you <EOS>
```

`collate_fn` 先在补 PAD 之前记录每条中文序列的真实长度，再把当前 Batch 补到该 Batch 的最大长度：

```text
source_padded：[batch_size, src_len]
target_padded：[batch_size, tgt_len]
src_lengths：  [batch_size]
```

`src_lengths` 用于 `pack_padded_sequence`。仅仅让 PAD 的 Embedding 为零还不够，因为零向量进入 GRU 后仍会发生状态递推；Pack 才是真正跳过无效 PAD 时间步。

## 4. 构建模型 model.py

实际训练配置为：

```text
Embedding：128 维
Encoder：  2 层双向 GRU，每个方向 hidden_size=64
Decoder：  2 层单向 GRU，hidden_size=128
Dropout：  0.3，作用在 GRU 层与层之间
```

整体流程如下：

```text
中文编号 [B,S]
→ 中文 Embedding [B,S,128]
→ 双向 Encoder GRU
→ 逐位置 output [B,S,128]
→ 原始 hidden [4,B,64]
→ 每层正反向拼接 [2,B,128]
→ 作为 Decoder 初始 hidden
→ Decoder 每步输入一个英文 Token [B]
→ 单步 logits [B,V_en]
→ 堆叠为完整 logits [B,T-1,V_en]
```

双向 Encoder 原始 hidden 的排列为：

```text
第 0 层正向、第 0 层反向、第 1 层正向、第 1 层反向
```

代码分别用 `hidden[0::2]`、`hidden[1::2]` 取出正向和反向状态，再沿特征维拼接：

```text
[2,B,64] + [2,B,64] → [2,B,128]
```

这里拼接的是**同一层的正向与反向状态**。正向最终状态读到原句末尾，反向最终状态读到原句开头；不应笼统地说两个方向都“对应句尾”。

Encoder 返回的两个结果含义不同：

| 返回值 | 含义 | 当前用途 |
| --- | --- | --- |
| `output [B,S,128]` | Encoder 顶层在各源位置的表示 | 已计算并返回，但当前 Seq2Seq 丢弃 |
| `hidden [2,B,128]` | 各层两个方向读完整句后的最终状态 | 直接初始化 Decoder |

Decoder 每一步返回 `(logits, hidden)`：logits 用于预测下一个 Token，更新后的 hidden 继续传给下一步。当前状态直接交接还隐含要求 Encoder 与 Decoder 层数相同；代码显式检查了 hidden 宽度，但没有单独检查层数。

模型初始化使用 Xavier、正交初始化等策略。由于通用初始化会覆盖 Embedding 中 PAD 行的默认零向量，`build_model()` 在初始化后再次把中英文 PAD 行清零。

## 5. 训练 train.py

训练流程如下：

```text
设置随机种子和设备
→ 创建 DataLoader 与词表
→ 构建并初始化模型
→ 逐 Batch 前向传播
→ 交叉熵损失（忽略 PAD）
→ 反向传播
→ 梯度裁剪
→ AdamW 更新参数
→ 验证集 loss
→ Early Stopping 与保存最佳权重
```

主要训练参数：

```text
batch_size：     64
learning_rate：  0.001
max_epochs：     50
gradient_clip：  1.0
early_stopping： 8 轮未改善
optimizer：      AdamW（weight_decay=1e-4）
```

交叉熵计算前，将 Batch 和时间维展开：

```text
logits：[B,T-1,V_en] → [B×(T-1),V_en]
target：[B,T] 的第 1 位之后 → [B×(T-1)]
```

`CrossEntropyLoss` 直接接收 logits，内部完成等价于 LogSoftmax 和负对数似然的计算，因此训练前不需要手动添加 Softmax。`ignore_index=<PAD>` 使补齐位置不参与损失。

训练阶段的 Teacher Forcing 比率从 1.0 线性下降并接近 0.5。前期主要使用正确英文前文帮助收敛，后期逐渐让模型接收自己的预测，尝试减轻训练与推理之间的不一致。按当前 50 轮循环的下标公式，最后一轮实际为 0.51，而不是恰好 0.50。

验证阶段固定使用 `teacher_forcing_ratio=1.0`。因此 `val_loss` 衡量的是：

```text
给定正确英文前文时，预测下一个 Token 的能力
```

它不能直接代表模型完全依靠自己生成时的翻译质量。训练 loss 后期包含模型自己的错误前缀和 Dropout，验证 loss 则使用正确前缀并关闭 Dropout，所以两条曲线也不是严格相同条件。

训练过程使用 TensorBoard 记录 train/val loss、Teacher Forcing 比率和翻译示例；每 5 轮使用真实贪心生成展示验证句。验证 loss 创造新低时保存 `weights/best.pt`，连续 8 轮不改善则提前停止。当前 checkpoint 只保存 `state_dict`，没有绑定词表、配置、epoch 和优化器状态，因此权重必须与训练时的词表编号和模型配置配套使用。

## 6. 推理与评估 utils.py / predict.py / evaluate.py

独立翻译流程如下：

```text
输入中文字符串
→ 中文分词与编号，并添加 <EOS>
→ Encoder 编码一次
→ Decoder 首步输入 <SOS>
→ 每一步选择 logits 最大的 Token
→ 把预测 Token 和更新后的 hidden 传入下一步
→ 遇到 <EOS> 或达到 50 步时停止
→ 英文词表 decode 并拼接成字符串
```

训练和推理的关键区别：

| 阶段 | Decoder 下一步输入 | 是否需要参考英文 |
| --- | --- | --- |
| 训练 | 按 Teacher Forcing 比率选择真值或模型预测 | 需要 |
| 验证 loss | 始终使用正确英文前文 | 需要 |
| 贪心推理 | 始终使用模型上一步预测 | 不需要 |

贪心解码只需要 `argmax(logits)`，不必先做 Softmax，因为 Softmax 不会改变最大值所在的位置。`model.eval()` 关闭 Dropout，`torch.no_grad()` 关闭梯度记录；工具函数结束后还会恢复模型进入函数之前的 train/eval 状态。

`predict.py` 加载中英文词表和 `best.pt` 后进入交互翻译。需要注意：`SAVE_DIR="weights"` 是相对当前启动目录的路径，不是固定相对于脚本文件；换工作目录运行时可能找不到权重。

`evaluate.py` 不使用正确英文前缀，而是逐句调用真实贪心翻译，再计算带平滑的语料 BLEU-4。因此它比教师强制 `val_loss` 更接近实际生成质量，但 BLEU 仍不能完整判断语义是否正确。NLTK `corpus_bleu` 返回的是 0～1 范围的小数；若按 0～100 展示，应显式乘以 100，不能混用两个尺度。

## 7. 实验结果与局限性

本次对比记录中：

```text
0914 版：val_loss = 3.2554
0918 版：val_loss = 2.8050
```

0918 在 10 条人工对照句中的表现明显优于 0914：按表格标记实际为 4 条正确、3 条部分命中、3 条错误。常见短句如“你叫什么名字？”、“今天天气很好。”、“我在学英语。”和“这个多少钱？”已经能够得到基本正确的译文。

同时仍然存在以下真实失败现象：

- “请帮我找一下我的钱包。”保留了 `help`、`my wallet`，但丢失“找”的动作，语法也不完整；
- “汤姆不想让玛丽失望。”保留了人名和否定框架，却把“让玛丽失望”错误地变成“让玛丽帮助玛丽”；
- “富士山顶盖满了雪。”把实体 `Mt. Fuji` 翻出，但将“被雪覆盖”错误地生成成“被偷了”；
- “明天我要去北京。”保留了 `Tomorrow` 和“我要去”的框架，但丢失“北京”并重复 `tomorrow`。

这些现象说明 0918 已经比 0914 更能保留源句的关键词和常见句型，但对动作、角色关系、地点和修饰关系仍不稳定。英文表面流畅不等于忠实于中文原文。

还需要注意以下评估边界：

1. 这 10 句是人工展示样例，不能当作模型的全局准确率；
2. 两版的数据划分、词表、模型和 loss 汇总方式并不完全相同，不能只用两个 `val_loss` 数字计算翻译能力提升百分比；
3. 当前 2,639 条验证数据中，有 394 条的原始中文字符串也出现在训练集，但没有完全相同的中英文句对跨集合重复；验证集并不等于全部都是从未见过的中文表达；
4. 当前 Encoder 虽然是双向 GRU，整句信息仍主要通过固定大小的最终 hidden 传给 Decoder；
5. Teacher Forcing 逐渐降低只能缓解 exposure bias，不能完全消除推理时的错误累积；
6. 词级词表会把不同未登录词统一映射为 `<UNK>`，具体语义仍可能丢失；
7. 贪心解码只保留每一步当前最优词，局部选择错误后无法回退。

因此，更准确的结论是：0918 的完整 GRU Seq2Seq 方案在这些课堂样例上取得了明显进步，但还没有获得稳定的开放句子翻译能力。

## 8. 引出下一节课的 Attention 机制

当前 Encoder 已经计算并返回所有源位置的表示：

```text
encoder_output：[B,S,128]
```

但是当前 `Seq2Seq.forward()` 和贪心推理都只使用最终 hidden，把 `encoder_output` 丢弃了。Decoder 生成每个英文词时，不能重新查看具体中文位置。

Attention 的核心改进是：

```text
Decoder 当前 hidden 作为 Query
+ Encoder 各位置 output 作为 Key / Value
→ 计算源序列各位置的注意力权重
→ 加权求和得到当前 context
→ 结合 context 预测下一个英文词
```

例如：

```text
生成 weather  → 重点关注“天气”
生成 wallet   → 重点关注“钱包”
生成 Beijing  → 重点关注“北京”
```

这样，Decoder 不必只依赖一个固定长度 hidden 保存整句话，而可以在每一步动态决定“当前应该看中文的哪个位置”。加入 Attention 后仍然保留 GRU 的时间递推；Transformer 则进一步用 Self-Attention 替代序列方向上的循环计算。

0914 解决了基础 Seq2Seq 的完整流程，0918 通过双向 GRU、门控状态和更完整的训练推理工程提高了效果；Attention 接下来要解决的是：**生成每一个词时，怎样重新利用源句中最相关的位置。**
